Our package provides data access in a Python programming environment.

Here, we will start a Clustering analysis for the Pancreatic ductal adenocarcinoma (pdac).

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# from gpnotebook.tools.standard_imports import *
import os, re,sys
import yaml
import pandas as pd
import numpy as np


In [2]:
# project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/PDAC_P_PDC000271"
project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/BRCA_P_PDC000121"
data_dir = os.path.join(project_dir,"matrix")
meta_dir = os.path.join(project_dir,"meta")
job_dir = os.path.join(project_dir,"precomputed","cluster")
if not os.path.exists(job_dir):
    os.mkdir(job_dir)

In [3]:
data_path = os.path.join(data_dir, "DIG_nglycoform-peptide_matrix-abundances-MD_norm.tsv")
data_df = pd.read_csv(data_path,sep="\t", index_col = [0,1,2,3])
data_df

Intensity.Reference  \
Site                                               Gene    Sequence                              Glycan                            
ENSP00000220847@150;ENSP00000338269@150;ENSP000... PHF20L1 AAAAAAAKNKTGSKPR                      N7H4F2S1G0            12.457372   
ENSP00000300107@452                                CLPX    AAAAADLANRSGESNTHQDIEEK               N4H4F3S1G0            14.878940   
ENSP00000275072@295                                PM20D2  AAALASGCTVEIKGGAHDYYNVLPNK            N6H4F0S0G0            14.318357   
ENSP00000431932@33;ENSP00000385235@64              LARGE2  AAALDGDPGAGPGDHNRSDCGPQPPPPPK         N3H5F1S1G0            12.184160   
ENSP00000355601@95                                 EGLN1   AAARRDNASGDAAK                        N5H6F3S0G0            13.581372   
...                                                                                                                          ...   
ENSP00000310658@528;ENSP00000429969@557;ENSP000... SCUBE2  YVNLTCSSGK                            N2H5F0S0G0            15.305399   
                                                                                                 N4H3F0S2G0            13.140584   
ENSP00000386043@333                                LTBP1   YVQDQVAAPFQLSNHTGR                    N4H5F3S0G0            12.843528   
ENSP00000347041@108;ENSP00000496856@19             FMOD    YVYFQNNQITSIQEGVFDNATGLLWIALHGNQITSDK N6H7F0S0G0            11.717821   
ENSP00000359596@547                                CLCA2   YYTNNFITNLTFR                         N7H7F5S1G0            14.692397   

                                                                                                             11BR047_T_01  \
Site                                               Gene    Sequence                              Glycan                     
ENSP00000220847@150;ENSP00000338269@150;ENSP000... PHF20L1 AAAAAAAKNKTGSKPR                      N7H4F2S1G0           NaN   
ENSP00000300107@452                                CLPX    AAAAADLANRSGESNTHQDIEEK               N4H4F3S1G0     14.311066   
ENSP00000275072@295                                PM20D2  AAALASGCTVEIKGGAHDYYNVLPNK            N6H4F0S0G0     13.518575   
ENSP00000431932@33;ENSP00000385235@64              LARGE2  AAALDGDPGAGPGDHNRSDCGPQPPPPPK         N3H5F1S1G0     12.444701   
ENSP00000355601@95                                 EGLN1   AAARRDNASGDAAK                        N5H6F3S0G0     12.554488   
...                                                                                                                   ...   
ENSP00000310658@528;ENSP00000429969@557;ENSP000... SCUBE2  YVNLTCSSGK                            N2H5F0S0G0           NaN   
                                                                                                 N4H3F0S2G0           NaN   
ENSP00000386043@333                                LTBP1   YVQDQVAAPFQLSNHTGR                    N4H5F3S0G0           NaN   
ENSP00000347041@108;ENSP00000496856@19             FMOD    YVYFQNNQITSIQEGVFDNATGLLWIALHGNQITSDK N6H7F0S0G0           NaN   
ENSP00000359596@547                                CLCA2   YYTNNFITNLTFR                         N7H7F5S1G0           NaN   

                                                                                                             11BR043_T_01  \
Site                                               Gene    Sequence                              Glycan                     
ENSP00000220847@150;ENSP00000338269@150;ENSP000... PHF20L1 AAAAAAAKNKTGSKPR                      N7H4F2S1G0           NaN   
ENSP00000300107@452                                CLPX    AAAAADLANRSGESNTHQDIEEK               N4H4F3S1G0     14.918107   
ENSP00000275072@295                                PM20D2  AAALASGCTVEIKGGAHDYYNVLPNK            N6H4F0S0G0     14.593042   
ENSP00000431932@33;ENSP00000385235@64              LARGE2  AAALDGDPGAGPGDHNRSDCGPQPPPPPK         N3H5F1S1G0     12.546934   
ENSP00000355601@

In [4]:
meta_path= os.path.join(meta_dir, "BRCA_meta.txt")
meta_df = pd.read_csv(meta_path,sep="\t",header=[0,1])
meta_df

,case_id,Age,Sex,Tumor_Size_cm,Histologic_Grade,Tumor_necrosis,Path_Stage_pT,Path_Stage_pN,Stage,BMI,Tobacco_smoking_history,MAP3K1_mutation,GATA3_mutation,PIK3CA_mutation,TP53_mutation
,data_type,CON,BIN,CON,ORD,BIN,ORD,ORD,ORD,CON,ORD,BIN,BIN,BIN,BIN
0,01BR001,55,Female,NaN,NaN,NaN,pT2,pN1,Stage II,NaN,NaN,0,0,0,0
1,01BR015,35,Female,NaN,NaN,NaN,pT2,pN1,Stage II,NaN,NaN,0,0,1,1
2,01BR017,45,Female,NaN,NaN,NaN,pT3,pN1,Stage III,NaN,NaN,0,0,0,1
3,01BR018,66,Female,NaN,NaN,NaN,pT3,pN1,Stage III,NaN,NaN,0,0,0,1
4,01BR025,62,Female,NaN,NaN,NaN,pT3,pN1,Stage III,NaN,NaN,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,01BR008,48,Female,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,1
118,01BR009,64,Female,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,1
119,01BR010,65,Female,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,0


In [5]:
meta_cols = ['case_id','Sex','Stage']
meta2 = meta_df.loc[:,meta_cols]
meta2.columns = ['Sample.ID'] + meta_cols[1:]
meta2

,Sample.ID,Sex,Stage
0,01BR001,Female,Stage II
1,01BR015,Female,Stage II
2,01BR017,Female,Stage III
3,01BR018,Female,Stage III
4,01BR025,Female,Stage III
...,...,...,...
117,01BR008,Female,NaN
118,01BR009,Female,NaN
119,01BR010,Female,NaN
120,01BR020,Female,NaN


In [6]:
meta2.head(17)

,Sample.ID,Sex,Stage
0,01BR001,Female,Stage II
1,01BR015,Female,Stage II
2,01BR017,Female,Stage III
3,01BR018,Female,Stage III
4,01BR025,Female,Stage III
5,01BR026,Female,NaN
6,01BR027,Female,Stage II
7,01BR030,Female,Stage III
8,01BR031,Female,Stage III
9,01BR032,Female,Stage III


In [7]:
head_cols = ['Site', 'Gene', 'Sequence', 'Glycan', 'Intensity.Reference']
samples = [i for i in data_df.columns.values if i not in head_cols]
samples = [i for i in samples if i.split('_')[0] in list(meta2['Sample.ID']) and i.split('_')[1] == 'T']
len(samples)

130

In [8]:
rows = []
for sample in samples:
    key = sample.split('_')[0]
    row = meta2[meta2['Sample.ID']==key].iloc[0]
    row['Sample.ID'] = sample
    rows.append(row)
meta3 = pd.DataFrame(rows)

In [9]:
meta3.head(17)

,Sample.ID,Sex,Stage
71,11BR047_T_01,Female,Stage II
69,11BR043_T_01,Female,Stage II
72,11BR049_T_01,Female,Stage III
57,11BR023_T_01,Female,Stage II
99,18BR010_T_01,Female,Stage III
34,06BR003_T_01,Female,Stage II
84,11BR074_T_01,Female,Stage III
101,18BR017_T_01,Female,Stage I
2,01BR017_T_01,Female,Stage III
36,06BR006_T_02,Female,NaN


In [10]:
meta3 = meta3.replace(np.nan,'NA')

In [11]:
top_ann_data_path = os.path.join(job_dir,'top_ann_data.tsv')
meta3.to_csv(top_ann_data_path, sep="\t", index=False)

Top annotation settings.

In [12]:

top_ann_settings = {
    'Sex': {
        'Male': 'blue',
        'Female': 'red',
        'NA': 'grey',
    },
    'Stage': {
        'Stage I': 'blue',
        'Stage II': 'green',
        'Stage III': 'orange',
        'Stage IV': 'red',
        'NA': 'grey'
    },

}
top_ann_settings_path = os.path.join(job_dir,'top_ann_settings.yml')
with open(top_ann_settings_path,'w') as f:
    yaml.dump(top_ann_settings,f,default_flow_style=False)

In [13]:
data_df.head(2)

,,,,Intensity.Reference,11BR047_T_01,11BR043_T_01,11BR049_T_01,11BR023_T_01,18BR010_T_01,06BR003_T_01,11BR074_T_01,18BR017_T_01,01BR017_T_01,...,01BR023_T_17,11BR011_T_17,01BR020_T_17,20BR006_T_17,21BR010_T_17,09BR001_T_17,03BR011_T_17,11BR036_T_17,01BR010_T_17,pool_17
Site,Gene,Sequence,Glycan,,,,,,,,,,,,,,,,,,,,,
ENSP00000220847@150;ENSP00000338269@150;ENSP00000378788@150;ENSP00000378775@180;ENSP00000378784@176;ENSP00000378777@176,PHF20L1,AAAAAAAKNKTGSKPR,N7H4F2S1G0,12.457372,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENSP00000300107@452,CLPX,AAAAADLANRSGESNTHQDIEEK,N4H4F3S1G0,14.878940,14.311066,14.918107,13.361291,13.154153,15.046404,14.192254,13.312369,14.621667,13.831166,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
samples = meta3['Sample.ID'].to_list()

In [15]:
df2 = data_df.loc[:,samples].dropna()

In [16]:
df2.shape

(605, 130)

In [17]:
from scipy.stats import variation
rows = []
for index,row in df2.iterrows():
    rows.append([variation([np.power(2,i) for i in list(row)])])
cv_df = pd.DataFrame(rows,columns=['cv'],index= df2.index)

glycopeptides = cv_df[cv_df['cv']>0.25].index

data2 = df2[df2.index.isin(glycopeptides)]
glycopeptides =  [f'{site}@{gene}@{seq}@{glycan}' for site,gene,seq,glycan in glycopeptides]
data2.index = glycopeptides
tumor_expression_path = os.path.join(job_dir,'expression_data.tsv')
data2.to_csv(tumor_expression_path,sep='\t',index=True)

In [18]:
data2.shape

(591, 130)

Extract tumor samples from glycopeptide expression data based on pathological status,

calculates the coefficient of variation (CV) for each glycopeptide, selects glycopeptides with CV greater than 0.25.

Map glcopeptides with cv>0.25 in tumor patients with glycan type.

In [19]:
import re,os, sys

def decide_glycan_type(g):
    m = re.finditer("([A-Z])([\d]+)", g)
    y = [(i.group(1), int(i.group(2))) for i in m]
    d = dict(y)
    glycan_type = "Other"
    if d["N"] == 2 and d["H"] >= 5 and d["F"] == 0 and d["S"] == 0 and d["G"] == 0:
        glycan_type = "HM"
    elif d["N"] >= 2 and d["H"] >= 3 and d["F"] > 0 and d["S"] == 0:
        glycan_type = "only_F"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] == 0:
        glycan_type = "only_S"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] > 0:
        glycan_type = "F+S"
    return glycan_type


In [20]:
# left annotation
# from gpnotebook.tools.glycan import decide_glycan_type

glycan_type_map = dict(zip(glycopeptides,[decide_glycan_type(i) for i in glycopeptides]))
  
left_ann_data_path =  os.path.join(job_dir,'left_annotation_data.tsv')
rows = []
for i in glycan_type_map:
    rows.append([i,glycan_type_map[i]])
left_ann_data = pd.DataFrame(rows,columns=['Glycopeptide','GlycanType'])
left_ann_data.to_csv(left_ann_data_path,sep="\t",index=False)

In [21]:
left_ann_data

,Glycopeptide,GlycanType
0,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N3H...,F+S
1,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N5H...,F+S
2,ENSP00000273784@166;ENSP00000393887@165@AHSG@A...,only_S
3,ENSP00000273784@166;ENSP00000393887@165@AHSG@A...,only_S
4,ENSP00000273784@166;ENSP00000393887@165@AHSG@A...,F+S
...,...,...
586,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
587,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
588,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
589,ENSP00000261590@458@DSG2@YVQNGTYTVK@N4H5F1S1G0,F+S


Map glycan types with colors.

In [22]:

# left annotation settings, including color, order
left_ann_settings_path = os.path.join(job_dir,'left_annotation_settings.yml')
left_ann_settings = {
    "glycan_type_index" :{
    "HM": 1,
    "only_F":2,
    "only_S":3,
    "F+S":4,
    "Other":5
    },
    "glycan_type_color" : {
        "HM": 'green',
    "only_F": 'red',
    "only_S": 'purple',
    "F+S": 'orange',
    "Other": 'grey'
}
}
with open(left_ann_settings_path,'w') as f:
    yaml.dump(left_ann_settings,f,default_flow_style=False)
    

Parameters for NMF clustering.

In [23]:
nmf_parameters_path = os.path.join(job_dir, 'nmf_parameters.yml')
nmf_parameters = {
    'k_range': {
        'min': 3,
        'max': 5,
    },
    'test':{
        'nruns': 50
    },
    'opt_k':{
        'nruns': 500,
        'predefined': 0,
        'value': 4,
        'feature_prob': 0.8
    }
}
with open(nmf_parameters_path,'w') as f:
    yaml.dump(nmf_parameters,f,default_flow_style=False)

Generate a YAML configuration file (nmf_configs.yml) containing paths to various data required for NMF clustering.

In [24]:
config_data = {
    'input': {
        'expression_data': tumor_expression_path,
        'left_annotation_data': left_ann_data_path ,
        'left_annotation_settings': left_ann_settings_path,
        'top_annotation_data': top_ann_data_path,
        'top_annotatin_settings': top_ann_settings_path,
        'nmf_parameters': nmf_parameters_path
    },
    'output':{
        'out_dir': job_dir
    }
}
nmf_configs_path = os.path.join(job_dir,'nmf_configs.yml')
with open(nmf_configs_path,'w') as f:
    yaml.dump(config_data,f,default_flow_style=False)